In [ ]:
!pip install datasets rouge_score nltk

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1cdbf3f6c4aa7ef2944fe66b195cff852b8a92d7bf814d8975d4e77a3255eeac
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


cuda


In [ ]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

train_data = dataset["train"].select(range(5000))
val_data   = dataset["validation"].select(range(500))
test_data  = dataset["test"].select(range(500))

In [ ]:
from collections import Counter
import re

def tokenize(text):
    text = text.lower()
    return re.findall(r"\w+", text)

counter = Counter()

for row in train_data:
    counter.update(tokenize(row["article"])[:500])
    counter.update(tokenize(row["highlights"]))

vocab = {
    "<pad>":0,
    "<sos>":1,
    "<eos>":2,
    "<unk>":3
}

for word, freq in counter.items():
    if freq >= 3:
        vocab[word] = len(vocab)

ivocab = {v:k for k,v in vocab.items()}

print("Vocab size:", len(vocab))

Vocab size: 28541


In [ ]:
MAX_ART = 400
MAX_SUM = 100

def encode(text, max_len):
    toks = tokenize(text)[:max_len-2]
    ids = [1]  # sos
    ids += [vocab.get(w,3) for w in toks]
    ids += [2] # eos

    while len(ids) < max_len:
        ids.append(0)
    return ids[:max_len]

In [ ]:
class CNNDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        src = torch.tensor(encode(row["article"], MAX_ART))
        trg = torch.tensor(encode(row["highlights"], MAX_SUM))
        return src, trg

In [ ]:
train_loader = DataLoader(CNNDataset(train_data), batch_size=16, shuffle=True)
val_loader   = DataLoader(CNNDataset(val_data), batch_size=16)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb=128, hid=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)

    def forward(self, x):
        emb = self.emb(x)
        outputs, (h,c) = self.lstm(emb)
        return outputs, h, c

In [ ]:
class Attention(nn.Module):
    def __init__(self, hid):
        super().__init__()
        self.W = nn.Linear(hid*3, hid)
        self.v = nn.Linear(hid,1,bias=False)

    def forward(self, hidden, encoder_outputs):
        B,T,H = encoder_outputs.shape

        hidden = hidden.unsqueeze(1).repeat(1,T,1)
        energy = torch.tanh(self.W(torch.cat((hidden, encoder_outputs), dim=2)))
        scores = self.v(energy).squeeze(2)
        attn = torch.softmax(scores, dim=1)

        context = torch.bmm(attn.unsqueeze(1), encoder_outputs)
        return context.squeeze(1)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb=128, hid=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.attn = Attention(hid)
        self.lstm = nn.LSTM(emb + hid*2, hid, batch_first=True)
        self.fc = nn.Linear(hid + hid*2, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(1)
        emb = self.emb(x)
        context = self.attn(hidden, encoder_outputs).unsqueeze(1)

        inp = torch.cat((emb, context), dim=2)
        output, (h,c) = self.lstm(inp, (hidden.unsqueeze(0), cell.unsqueeze(0)))

        pred = self.fc(torch.cat((output.squeeze(1), context.squeeze(1)), dim=1))

        return pred, h.squeeze(0), c.squeeze(0)

seq2seq wrapper class

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.enc = Encoder(vocab_size)
        self.dec = Decoder(vocab_size)

    def forward(self, src, trg):
        B,T = trg.shape
        vocab_size = len(vocab)

        outputs = torch.zeros(B,T,vocab_size).to(device)

        enc_out, h, c = self.enc(src)

        h = h[-2] + h[-1]
        c = c[-2] + c[-1]

        x = trg[:,0]

        for t in range(1,T):
            out,h,c = self.dec(x,h,c,enc_out)
            outputs[:,t] = out
            x = trg[:,t]

        return outputs

In [ ]:
model = Seq2Seq(len(vocab)).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    total=0

    for src,trg in train_loader:
        src,trg = src.to(device), trg.to(device)

        optimizer.zero_grad()
        out = model(src,trg)

        loss = criterion(
            out[:,1:].reshape(-1,len(vocab)),
            trg[:,1:].reshape(-1)
        )

        loss.backward()
        optimizer.step()
        total += loss.item()

    print("Epoch",epoch+1,"Loss:",total/len(train_loader))

Epoch 1 Loss: 7.77307421559343
Epoch 2 Loss: 6.895157995315405
Epoch 3 Loss: 6.152680311720973
Epoch 4 Loss: 5.397183735149737
Epoch 5 Loss: 4.612566883190752


In [ ]:
def summarize(text):
    model.eval()

    src = torch.tensor(encode(text, MAX_ART)).unsqueeze(0).to(device)

    with torch.no_grad():
        enc_out,h,c = model.enc(src)
        h = h[-2] + h[-1]
        c = c[-2] + c[-1]

        x = torch.tensor([1]).to(device)
        result=[]

        for _ in range(MAX_SUM):
            out,h,c = model.dec(x,h,c,enc_out)
            pred = out.argmax(1).item()

            if pred==2:
                break

            result.append(ivocab.get(pred,""))
            x = torch.tensor([pred]).to(device)

    return " ".join(result)

In [ ]:
scores_r1 = []
scores_r2 = []
scores_rl = []

for i in range(100):
    pred = summarize(test_data[i]["article"])
    ref  = test_data[i]["highlights"]

    score = scorer.score(ref, pred)

    scores_r1.append(score["rouge1"].fmeasure)
    scores_r2.append(score["rouge2"].fmeasure)
    scores_rl.append(score["rougeL"].fmeasure)

print("Avg ROUGE-1:", sum(scores_r1)/len(scores_r1))
print("Avg ROUGE-2:", sum(scores_r2)/len(scores_r2))
print("Avg ROUGE-L:", sum(scores_rl)/len(scores_rl))

Avg ROUGE-1: 0.10095081880147498
Avg ROUGE-2: 0.009543300989561158
Avg ROUGE-L: 0.08223368567782838
